# bank_distill: relational and contrastive distillation against a bank of cached teacher vectors

A batch of 128 random texts holds almost none of a text's real neighbours (measured: 1.2% of its top-10), so the in-batch relation terms (Gram, ranking, listwise) only compare it with far-away texts. The teacher vectors of all 200K texts are cached, so each step can also compare every text with a bank of 65,536 cached teacher vectors of random training texts (redrawn every 200 steps; no extra encoding, one matrix product per teacher). Code: `src/turkish_distill/bank.py`, `loss.bank_*` in `losses.py`.

- (A) relational, adapted from RRD / SEED / CompRess (vision): the teacher's distribution of the text over the bank, softmax(t . bank / 0.02), is the target of the student's head output, KL(teacher || student). Same temperature on both sides, so the optimum agrees with the vector term.
- (B) contrastive, adapted from CKD (DistilCSE) / CRD: InfoNCE, the student's head output must be closer to its own teacher vector than to all 65,536 bank vectors (temperature 0.05).

Both are added to the usual Jasper loss (cosine + Gram + ranking), per teacher, the text itself is kept out of its own bank.

Runs (1k-base student, BGE-M3 + plain 8K-embed, 200K mix, lr 1e-4, 5400 steps, seed 42):
1. `bank_kl_s5400`: (A), weight 1
2. `bank_kl_nce_s5400`: (A) + (B), weights 1 / 1

Reference: the plain pair `scale_two_lr1e4_mix200k_s5400`, TR-MTEB 67.16 (seed 43: 67.25); only differences above 0.3 are read. Logs: `train/bank_kl/<teacher>`, `train/bank_nce/<teacher>`, `train/bank_acc/<teacher>` (share of texts whose own teacher vector beats the whole bank) next to the usual terms; read the term sizes after ~500 steps: if a bank term is 10x the others, its weight is too high. Re-running after a disconnect resumes; the last cell releases the runtime. Delete this notebook when the results are in.


In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive')

REPO_URL = 'https://github.com/mericberktas/bitirme.git'
BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/bitirme'
REPO_DIR = '/content/repo'

os.makedirs(DRIVE_ROOT, exist_ok=True)
# Models are downloaded to the LOCAL disk each session. Do not put the HF cache on Drive:
# symlinks and large files are not stored reliably there (it produced corrupted weights).
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch --quiet && git -C {REPO_DIR} checkout --quiet {BRANCH} && git -C {REPO_DIR} pull --quiet
else:
    !git clone --quiet --branch {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install --quiet -e .
!git log -1 --oneline

In [ ]:
# --- Make sure the environment is usable (safe to re-run; covers a restarted runtime) ---
import importlib.util, os, subprocess, sys

DRIVE_ROOT = globals().get('DRIVE_ROOT', '/content/drive/MyDrive/bitirme')
REPO_DIR = globals().get('REPO_DIR', '/content/repo')

if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
if not os.path.isdir(f'{REPO_DIR}/src'):
    raise RuntimeError('Repo not found: run the Setup cell (section 1) first.')
os.chdir(REPO_DIR)  # config paths below are relative to the repo root

if importlib.util.find_spec('turkish_distill') is None:
    print('turkish_distill is not installed in this runtime: installing it now...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-e', REPO_DIR], check=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec('turkish_distill') is None:
        sys.path.insert(0, f'{REPO_DIR}/src')
        os.environ['PYTHONPATH'] = f'{REPO_DIR}/src' + os.pathsep + os.environ.get('PYTHONPATH', '')

os.environ.setdefault('HF_HOME', '/content/hf_cache')  # local disk, never Drive
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception as exc:
        print('HF_TOKEN not set (only needed for gated models):', exc)

# Runs of this notebook, one after another (all: two-teacher config, 1k-base student, 200K mix, lr 1e-4, 5400 steps, seed 42).
# (name, config, extra overrides). A later override wins.
CONFIG = 'configs/stage1_two_bge_m3_modernbert_tr_embed.yaml'
BASE = ["data.corpus_path=/content/drive/MyDrive/bitirme/data/mix/mix_v2_200k.jsonl", "train.max_steps=5400", "train.lr=0.0001"]
BANK = ['loss.bank_size=65536', 'loss.bank_refresh_every=200']
RUNS = [
    ('bank_kl_s5400', CONFIG, BANK + ['loss.bank_kl_weight=1.0']),
    ('bank_kl_nce_s5400', CONFIG, BANK + ['loss.bank_kl_weight=1.0', 'loss.bank_nce_weight=1.0']),
]

import glob, os

for name, config, extra in RUNS:
    run_dirs = sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/'))
    if any(os.path.isdir(d + 'final') for d in run_dirs):
        print(f'\n########## {name}: already finished, skipped')
        continue
    run_id_arg = f'--run-id {os.path.basename(run_dirs[-1].rstrip("/"))}' if run_dirs else ''   # resume after a disconnect
    overrides = ' '.join(f'--override {o}' for o in [f'name={name}'] + BASE + extra)
    print(f'\n########## {name} {run_id_arg or "(new run)"}')
    get_ipython().system(f'python -m turkish_distill.train --config {config} {run_id_arg} {overrides}')

print('\nEval presets (paste into an eval notebook):')
for name, config, extra in RUNS:
    for d in sorted(glob.glob(f'{DRIVE_ROOT}/runs/{name}/*/final')):
        print(f"    'd_{name}': dict(model=f'{{DRIVE_ROOT}}/runs/{name}/{os.path.basename(os.path.dirname(d))}/final'),")


In [ ]:
from google.colab import runtime
runtime.unassign()